================================================================================
NOTEBOOK 1 — Build ALL Datasets
================================================================================
Construit 3 datasets à partir d'un CSV de visites médicales :

  ┌─────────────────┬───────────────────────────┬───────────────────────────────┐
  │ Dataset         │ Modèle cible              │ Champs extraits               │
  ├─────────────────┼───────────────────────────┼───────────────────────────────┤
  │ BIO (NER)       │ CamemBERT NER + Flair     │ nom_medecin, specialite,      │
  │                 │                           │ medicament, gadget            │
  ├─────────────────┼───────────────────────────┼───────────────────────────────┤
  │ QA (SQuAD)      │ QAmemberta                │ objectif, message_cle,        │
  │                 │                           │ reponse, action, commentaire  │
  ├─────────────────┼───────────────────────────┼───────────────────────────────┤
  │ Classification  │ CamemBERT Classifier      │ type_visite, niveau_interet   │
  └─────────────────┴───────────────────────────┴───────────────────────────────┘

Stratégie de split : GroupShuffleSplit par `nom_medecin` → pas de fuite de données.
Sortie : MyDrive/medical_project/datasets/

Dépendances :
    pip install pandas scikit-learn numpy matplotlib seaborn
================================================================================

─── IMPORTS ──────────────────────────────────────────────────────────────────

In [ ]:
import json
import re
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
# Modifier ces chemins selon votre environnement
CSV_PATH   = '/content/drive/MyDrive/rapport_data_fusion_sans_lignes_double.csv'
OUT_BASE   = '/content/drive/MyDrive/medical_project/datasets'
GROUP_COL  = 'nom_medecin'   # Colonne de groupement pour éviter la fuite de données
SEED       = 42
TRAIN_SIZE = 0.70            # 70% train / 15% val / 15% test

# Colonnes texte candidates (la première trouvée dans le CSV est utilisée)
CONTEXT_COLS = [
    'context_qamemberta',
    'resume_visite_diversifie',
    'resume_visite_bien_structure',
    'resume_visite_moyen',
]

# Valeurs considérées comme vides (normalisation)
EMPTY_VALS = {'', 'nan', 'none', 'null', 'n/a', 'non renseigné', 'nr', '-'}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 0 — UTILITAIRES COMMUNS

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def norm(x: object) -> str:
    """Convertit n'importe quelle valeur en chaîne propre (strip, NaN → '')."""
    return '' if pd.isna(x) else str(x).strip()


def is_empty(x: object) -> bool:
    """Retourne True si la valeur est considérée comme vide/manquante."""
    return norm(x).lower() in EMPTY_VALS

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 1 — CHARGEMENT ET NETTOYAGE DU CSV

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def load_and_clean_csv(csv_path: str, context_cols: list, group_col: str) -> tuple:
    """
    Charge le CSV source, sélectionne la colonne texte et nettoie les données.

    Retourne:
        df        (DataFrame nettoyé)
        text_col  (nom de la colonne texte sélectionnée)
    """
    print("─" * 60)
    print("ÉTAPE 1 : Chargement du CSV")
    print("─" * 60)

    df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
    print(f"  CSV chargé : {len(df)} lignes, {len(df.columns)} colonnes")

    # Sélection de la meilleure colonne texte disponible
    text_col = next((c for c in context_cols if c in df.columns), None)
    assert text_col, f"Aucune colonne texte trouvée. Colonnes disponibles : {list(df.columns)}"
    print(f"  Colonne texte sélectionnée : {text_col}")

    # Nettoyage : suppression des NaN, doublons et textes vides
    df[text_col] = df[text_col].fillna('').astype(str).str.strip()
    df = df[df[text_col] != ''].drop_duplicates(subset=[text_col]).reset_index(drop=True)
    print(f"  Après nettoyage : {len(df)} lignes")

    # Vérification de la colonne de groupement
    assert group_col in df.columns, f"Colonne de groupe '{group_col}' introuvable dans le CSV."
    print(f"  Nombre de médecins uniques : {df[group_col].nunique()}")

    return df, text_col

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 2 — SPLIT TRAIN / VAL / TEST

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def split_by_group(df: pd.DataFrame, group_col: str,
                   train_size: float = 0.70, seed: int = 42) -> tuple:
    """
    Découpe le dataset en train / val / test en respectant les groupes (médecins).

    Pourquoi GroupShuffleSplit ?
    → Empêche qu'un même médecin apparaisse dans train ET val/test,
      ce qui simulerait une fuite de données et gonflerait artificiellement les métriques.

    Split effectif : 70% train | 15% val | 15% test
    """
    print("\n─" * 60)
    print("ÉTAPE 2 : Split Train / Val / Test (par médecin)")
    print("─" * 60)

    groups = df[group_col].astype(str)

    # Premier split : train vs (val + test)
    gss1 = GroupShuffleSplit(n_splits=1, test_size=1 - train_size, random_state=seed)
    train_idx, temp_idx = next(gss1.split(df, groups=groups))
    train_df = df.iloc[train_idx].copy()
    temp_df  = df.iloc[temp_idx].copy()

    # Deuxième split : val vs test (50/50 du reste)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    dev_idx, test_idx = next(gss2.split(temp_df, groups=temp_df[group_col].astype(str)))
    val_df  = temp_df.iloc[dev_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    print(f"  Train : {len(train_df)} lignes | Val : {len(val_df)} | Test : {len(test_df)}")
    print(f"  Split par médecin → aucune fuite de données ✅")

    return train_df, val_df, test_df

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 3 — DATASET NER (BIO)
# Destiné à : CamemBERT NER + Flair
# Entités   : MEDECIN, SPECIALITE, MEDICAMENT, GADGET
# Format    : Un token par ligne, tag BIO — corpus Flair
# Pourquoi NER ? Ces entités sont courtes et précises (1-4 mots).
#               Le modèle étiquette chaque token → idéal pour les noms propres,
#               médicaments et spécialités médicales.

════════════════════════════════════════════════════════════════════════════════

Correspondance entité → colonne CSV

In [ ]:
NER_ENTITY_TO_COL = {
    'MEDECIN':    'nom_medecin',
    'SPECIALITE': 'specialite_medecin',
    'MEDICAMENT': 'medicament',
    'GADGET':     'gadget',
}


def find_spans(text: str, value: str, label: str) -> list:
    """
    Recherche toutes les occurrences non chevauchantes de `value` dans `text`.
    Retourne une liste de tuples (start, end, label).
    """
    value = norm(value)
    if is_empty(value) or len(value) < 3:
        return []

    spans = []
    for m in re.finditer(re.escape(value), text, re.IGNORECASE):
        s, e = m.start(), m.end()
        # Vérification des frontières de mot pour éviter les faux positifs
        if (s == 0 or not text[s-1].isalnum()) and (e == len(text) or not text[e].isalnum()):
            spans.append((s, e, label))
    return spans


def remove_overlaps(spans: list) -> list:
    """
    Supprime les spans qui se chevauchent en privilégiant le plus long.
    Retourne les spans triés par position de début.
    """
    spans = sorted(spans, key=lambda x: (x[0], -(x[1] - x[0])))
    out = []
    for s, e, l in spans:
        # Conserver seulement si aucun overlap avec les spans déjà acceptés
        if not any(not (e <= fs or s >= fe) for fs, fe, _ in out):
            out.append((s, e, l))
    return sorted(out, key=lambda x: x[0])


def tokenize_offsets(text: str) -> list:
    """
    Tokenise le texte en conservant les offsets de chaque token.
    Retourne une liste de tuples (token, start, end).
    """
    return [
        (m.group(), m.start(), m.end())
        for m in re.finditer(r"\w+(?:[-']\w+)*|[^\w\s]", text, re.UNICODE)
    ]


def to_bio(text: str, spans: list) -> list:
    """
    Convertit le texte et ses spans en séquence BIO.
    B-LABEL = début d'entité, I-LABEL = intérieur, O = hors entité.
    """
    tokens = tokenize_offsets(text)
    result = []
    for tok, ts, te in tokens:
        tag = 'O'
        for s, e, label in spans:
            if ts >= s and te <= e:
                tag = ('B-' if ts == s else 'I-') + label
                break
        result.append((tok, tag))
    return result


def row_to_ner_sample(row: pd.Series, text_col: str) -> dict | None:
    """
    Convertit une ligne CSV en échantillon NER (tokens + tags BIO).
    Retourne None si le texte est vide ou si aucune entité n'est trouvée.
    """
    text = norm(row[text_col])
    if not text:
        return None

    # Recherche de tous les spans pour chaque type d'entité
    spans = []
    for label, col in NER_ENTITY_TO_COL.items():
        val = norm(row.get(col, ''))
        if not is_empty(val):
            spans.extend(find_spans(text, val, label))

    spans = remove_overlaps(spans)
    bio = to_bio(text, spans)
    tokens = [t for t, _ in bio]
    tags   = [l for _, l in bio]

    # Ignorer les échantillons sans aucune entité
    if not bio or all(t == 'O' for t in tags):
        return None

    return {'tokens': tokens, 'tags': tags}


def df_to_ner_samples(data_df: pd.DataFrame, text_col: str) -> list:
    """Applique `row_to_ner_sample` sur tout le DataFrame et retourne les valides."""
    samples = []
    for _, row in data_df.iterrows():
        s = row_to_ner_sample(row, text_col)
        if s:
            samples.append(s)
    return samples


def write_flair_corpus(samples: list, path: str) -> None:
    """
    Écrit les échantillons NER au format Flair (un token par ligne, ligne vide = fin de phrase).
    Format : <token> <tag>
    """
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for s in samples:
            for tok, tag in zip(s['tokens'], s['tags']):
                f.write(f"{tok} {tag}\n")
            f.write('\n')   # Séparateur de phrase


def build_ner_dataset(train_df, val_df, test_df, out_base: str, text_col: str) -> tuple:
    """
    Construit et sauvegarde le corpus NER complet (train / dev / test).
    Retourne les trois listes d'échantillons pour les statistiques.
    """
    print("\n─" * 60)
    print("PARTIE A : Création du dataset NER (BIO)")
    print("─" * 60)

    ner_train = df_to_ner_samples(train_df, text_col)
    ner_val   = df_to_ner_samples(val_df,   text_col)
    ner_test  = df_to_ner_samples(test_df,  text_col)

    print(f"  NER → Train: {len(ner_train)} | Val: {len(ner_val)} | Test: {len(ner_test)}")

    ner_dir = Path(out_base) / 'ner'
    write_flair_corpus(ner_train, ner_dir / 'train.txt')
    write_flair_corpus(ner_val,   ner_dir / 'dev.txt')
    write_flair_corpus(ner_test,  ner_dir / 'test.txt')
    print(f"  Corpus NER sauvegardé → {ner_dir} ✅")

    return ner_train, ner_val, ner_test

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 4 — DATASET QA (FORMAT SQuAD)
# Destiné à : QAmemberta
# Champs    : objectif_visite, message_cle, reponse, prochaine_action, commentaire
# Format    : JSON {id, question, context, answers: {text, answer_start}}
# Pourquoi QA ? Ces champs sont des textes longs (plusieurs phrases).
#               QA trouve le span exact répondant à la question → bien supérieur
#               à NER pour du texte narratif multi-tokens.

════════════════════════════════════════════════════════════════════════════════

Questions multiples par champ pour enrichir les variations d'apprentissage

In [ ]:
QA_FIELD_QUESTIONS = {
    'objectif_visite': [
        "Quel était l'objectif de cette visite médicale ?",
        "Pourquoi cette visite a-t-elle été effectuée ?",
        "Quelle était la finalité de la visite ?",
    ],
    'message_cle': [
        "Quel était le message clé transmis au médecin ?",
        "Quel message principal le délégué a-t-il communiqué ?",
        "Quel est le message central de cette visite ?",
    ],
    'reponse': [
        "Quelle a été la réponse du médecin ?",
        "Comment le médecin a-t-il réagi au message du délégué ?",
        "Qu'a dit le médecin en réponse à la présentation ?",
    ],
    'prochaine_action': [
        "Quelle est la prochaine action prévue suite à cette visite ?",
        "Que doit faire le délégué lors de la prochaine étape ?",
        "Quel est le plan d'action après cette visite ?",
    ],
    'commentaire_visite': [
        "Quel est le commentaire général sur cette visite ?",
        "Y a-t-il des remarques supplémentaires sur cette visite ?",
        "Quelles sont les notes complémentaires du délégué ?",
    ],
}


def find_answer_span(context: str, answer_text: str) -> dict | None:
    """
    Recherche la position de `answer_text` dans `context`.
    Essaie d'abord la correspondance exacte, puis partielle (5 premiers mots).
    Retourne {text, answer_start} ou None si introuvable.
    """
    answer_text = norm(answer_text)
    if is_empty(answer_text) or len(answer_text) < 3:
        return None

    # Correspondance exacte
    m = re.search(re.escape(answer_text), context, re.IGNORECASE)
    if m:
        return {'text': [context[m.start():m.end()]], 'answer_start': [m.start()]}

    # Correspondance partielle sur les 5 premiers mots
    words = answer_text.split()[:5]
    if len(words) >= 2:
        partial = ' '.join(words)
        m = re.search(re.escape(partial), context, re.IGNORECASE)
        if m:
            return {'text': [context[m.start():m.end()]], 'answer_start': [m.start()]}

    return None


def df_to_qa_samples(data_df: pd.DataFrame, text_col: str) -> list:
    """
    Génère les paires (question, context, answer) pour chaque ligne et chaque champ QA.
    Les exemples sans réponse trouvée sont marqués `is_impossible: True`.
    """
    samples = []
    uid = 0
    for _, row in data_df.iterrows():
        context = norm(row[text_col])
        if not context or len(context) < 20:
            continue
        for field, questions in QA_FIELD_QUESTIONS.items():
            answer_val = norm(row.get(field, ''))
            for question in questions:
                uid += 1
                if not is_empty(answer_val):
                    ans = find_answer_span(context, answer_val)
                    if ans:
                        samples.append({
                            'id': str(uid),
                            'question': question,
                            'context': context,
                            'answers': ans,
                            'is_impossible': False,
                        })
                        continue
                # Aucune réponse trouvée → exemple impossible (utile pour le modèle)
                samples.append({
                    'id': str(uid),
                    'question': question,
                    'context': context,
                    'answers': {'text': [], 'answer_start': []},
                    'is_impossible': True,
                })
    return samples


def build_qa_dataset(train_df, val_df, test_df, out_base: str, text_col: str) -> tuple:
    """Construit et sauvegarde le dataset QA au format SQuAD JSON."""
    print("\n─" * 60)
    print("PARTIE B : Création du dataset QA (format SQuAD)")
    print("─" * 60)

    qa_train = df_to_qa_samples(train_df, text_col)
    qa_val   = df_to_qa_samples(val_df,   text_col)
    qa_test  = df_to_qa_samples(test_df,  text_col)

    print(f"  QA → Train: {len(qa_train)} | Val: {len(qa_val)} | Test: {len(qa_test)}")

    qa_dir = Path(out_base) / 'qa'
    qa_dir.mkdir(parents=True, exist_ok=True)
    for name, data in [('train', qa_train), ('validation', qa_val), ('test', qa_test)]:
        with open(qa_dir / f'{name}.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Dataset QA sauvegardé → {qa_dir} ✅")

    return qa_train, qa_val, qa_test

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 5 — DATASET CLASSIFICATION
# Destiné à : CamemBERT Classifier
# Champs    : type_visite, niveau_interet
# Format    : CSV {text, label}
# Pourquoi Classification ? Ces champs sont CATÉGORIELS : le modèle lit le compte-
#             rendu entier et prédit une catégorie (pas un span textuel).
#             Ex : type_visite = {premiere, suivi, relance}
#                  niveau_interet = {1/5, 2/5, 3/5, 4/5, 5/5}

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def normalize_interet(val: str) -> str | None:
    """
    Normalise le niveau d'intérêt vers le format X/5.
    Gère les formes textuelles (ex: 'très faible' → '1/5') et numériques.
    """
    t = norm(val).lower()
    if is_empty(t):
        return None

    # Correspondances qualitatives
    QUAL = {
        'très faible': '1', 'faible': '2', 'moyen': '3',
        'bon': '4', 'élevé': '5', 'fort': '5', 'excellent': '5',
    }
    for label_q, v in QUAL.items():
        if label_q in t:
            return f'{v}/5'

    # Formes numériques : "3/5", "3 sur 5", "3"
    m = re.search(r'\b([1-5])\s*(?:/|sur)\s*5\b', t)
    if m:
        return f'{m.group(1)}/5'
    m = re.search(r'\b([1-5])\b', t)
    if m:
        return f'{m.group(1)}/5'

    return None


def build_clf_dataset(data_df: pd.DataFrame, field: str,
                      text_col: str, normalize_fn=None) -> pd.DataFrame:
    """
    Construit un DataFrame {text, label} pour un champ de classification donné.
    `normalize_fn` permet d'appliquer une normalisation sur l'étiquette brute.
    """
    rows = []
    for _, row in data_df.iterrows():
        text = norm(row[text_col])
        if not text or len(text) < 20:
            continue
        raw_label = norm(row.get(field, ''))
        if is_empty(raw_label):
            continue
        label = normalize_fn(raw_label) if normalize_fn else raw_label.lower().strip()
        if not label:
            continue
        rows.append({'text': text, 'label': label})
    return pd.DataFrame(rows)


def build_classification_dataset(train_df, val_df, test_df,
                                  out_base: str, text_col: str) -> dict:
    """Construit et sauvegarde les datasets de classification pour les deux champs."""
    print("\n─" * 60)
    print("PARTIE C : Création du dataset Classification")
    print("─" * 60)

    clf_dir = Path(out_base) / 'classification'
    clf_dir.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        tv = build_clf_dataset(split_df, 'type_visite',    text_col)
        ni = build_clf_dataset(split_df, 'niveau_interet', text_col, normalize_fn=normalize_interet)

        tv.to_csv(clf_dir / f'type_visite_{split_name}.csv',    index=False)
        ni.to_csv(clf_dir / f'niveau_interet_{split_name}.csv', index=False)

        counts[split_name] = {'type_visite': len(tv), 'niveau_interet': len(ni)}
        print(f"  {split_name}: type_visite={len(tv)} | niveau_interet={len(ni)}")

    print(f"  Dataset Classification sauvegardé → {clf_dir} ✅")
    return counts

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 6 — VISUALISATIONS ET COURBES D'ÉVALUATION

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def plot_dataset_statistics(train_df, val_df, test_df, text_col: str,
                             ner_train, ner_val, ner_test,
                             qa_train, qa_val, qa_test,
                             clf_counts: dict,
                             out_base: str) -> None:
    """
    Génère un tableau de bord complet de visualisations pour les 3 datasets.
    Sauvegarde la figure dans out_base/dataset_statistics.png
    """
    print("\n─" * 60)
    print("PARTIE D : Génération des statistiques et graphiques")
    print("─" * 60)

    sns.set_theme(style='whitegrid', palette='muted')
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('Statistiques des Datasets — Pipeline Médical NLP', fontsize=16, fontweight='bold', y=1.01)

    splits     = ['Train', 'Val', 'Test']
    colors_bar = ['#2196F3', '#4CAF50', '#FF9800']

    # ── Graphique 1 : Taille du split par dataset ──────────────────────────────
    ax = axes[0, 0]
    sizes = [len(train_df), len(val_df), len(test_df)]
    bars = ax.bar(splits, sizes, color=colors_bar, edgecolor='white', linewidth=1.5)
    ax.set_title('Taille des splits (lignes brutes)', fontweight='bold')
    ax.set_ylabel('Nombre de lignes')
    for bar, val in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha='center', fontweight='bold')

    # ── Graphique 2 : Distribution de la longueur des textes ──────────────────
    ax = axes[0, 1]
    for df_split, label, color in zip([train_df, val_df, test_df], splits, colors_bar):
        lengths = df_split[text_col].str.len()
        ax.hist(lengths, bins=30, alpha=0.6, label=label, color=color)
    ax.set_title('Distribution longueur des textes', fontweight='bold')
    ax.set_xlabel('Nombre de caractères')
    ax.set_ylabel('Fréquence')
    ax.legend()

    # ── Graphique 3 : Camembert des splits (proportion) ───────────────────────
    ax = axes[0, 2]
    total = len(train_df) + len(val_df) + len(test_df)
    wedges, texts, autotexts = ax.pie(
        sizes, labels=splits, colors=colors_bar,
        autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    )
    ax.set_title('Proportion des splits', fontweight='bold')

    # ── Graphique 4 : Échantillons NER par split ───────────────────────────────
    ax = axes[1, 0]
    ner_sizes = [len(ner_train), len(ner_val), len(ner_test)]
    bars = ax.bar(splits, ner_sizes, color=colors_bar, edgecolor='white', linewidth=1.5)
    ax.set_title('Dataset NER — Échantillons valides', fontweight='bold')
    ax.set_ylabel('Nombre d\'échantillons')
    for bar, val in zip(bars, ner_sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontweight='bold')

    # ── Graphique 5 : Distribution des tags NER dans le train set ─────────────
    ax = axes[1, 1]
    tag_counts: dict = {}
    for sample in ner_train:
        for tag in sample['tags']:
            if tag != 'O':
                entity = tag[2:]  # Retire "B-" ou "I-"
                tag_counts[entity] = tag_counts.get(entity, 0) + 1

    if tag_counts:
        ent_names = list(tag_counts.keys())
        ent_vals  = list(tag_counts.values())
        ent_colors = sns.color_palette('Set2', len(ent_names))
        bars = ax.barh(ent_names, ent_vals, color=ent_colors)
        ax.set_title('Distribution des entités NER (train)', fontweight='bold')
        ax.set_xlabel('Nombre d\'occurrences')
        for bar, val in zip(bars, ent_vals):
            ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                    str(val), va='center', fontweight='bold')

    # ── Graphique 6 : Échantillons QA + taux impossible ───────────────────────
    ax = axes[1, 2]
    qa_possible   = [sum(1 for s in split if not s['is_impossible']) for split in [qa_train, qa_val, qa_test]]
    qa_impossible = [sum(1 for s in split if s['is_impossible'])     for split in [qa_train, qa_val, qa_test]]
    x = np.arange(len(splits))
    width = 0.35
    ax.bar(x - width/2, qa_possible,   width, label='Répondables',  color='#2196F3')
    ax.bar(x + width/2, qa_impossible, width, label='Impossibles', color='#F44336')
    ax.set_title('Dataset QA — Répondables vs Impossibles', fontweight='bold')
    ax.set_ylabel('Nombre de questions')
    ax.set_xticks(x)
    ax.set_xticklabels(splits)
    ax.legend()

    # ── Graphique 7 : Taux de couverture QA par champ ─────────────────────────
    ax = axes[2, 0]
    field_coverage = {}
    for sample in qa_train:
        # Approximation du champ à partir de la question
        for field in QA_FIELD_QUESTIONS.keys():
            q_words = field.replace('_', ' ')
            if any(w in sample['question'].lower() for w in q_words.split()[:2]):
                field_coverage.setdefault(field, {'answerable': 0, 'total': 0})
                field_coverage[field]['total'] += 1
                if not sample['is_impossible']:
                    field_coverage[field]['answerable'] += 1
                break

    if field_coverage:
        fields = list(field_coverage.keys())
        rates  = [field_coverage[f]['answerable'] / max(field_coverage[f]['total'], 1) * 100
                  for f in fields]
        short_fields = [f.replace('_', '\n') for f in fields]
        bars = ax.bar(short_fields, rates, color=sns.color_palette('Blues_d', len(fields)))
        ax.set_title('Taux de couverture QA par champ (train)', fontweight='bold')
        ax.set_ylabel('% de questions répondables')
        ax.set_ylim(0, 110)
        for bar, val in zip(bars, rates):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold')

    # ── Graphique 8 : Distribution type_visite (train) ────────────────────────
    ax = axes[2, 1]
    clf_dir = Path(out_base) / 'classification'
    tv_path = clf_dir / 'type_visite_train.csv'
    if tv_path.exists():
        tv_df = pd.read_csv(tv_path)
        vc = tv_df['label'].value_counts()
        bars = ax.bar(vc.index, vc.values,
                      color=sns.color_palette('Set3', len(vc)))
        ax.set_title('Distribution type_visite (train)', fontweight='bold')
        ax.set_ylabel('Nombre d\'exemples')
        ax.tick_params(axis='x', rotation=20)
        for bar, val in zip(bars, vc.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(val), ha='center', fontweight='bold')

    # ── Graphique 9 : Distribution niveau_interet (train) ─────────────────────
    ax = axes[2, 2]
    ni_path = clf_dir / 'niveau_interet_train.csv'
    if ni_path.exists():
        ni_df = pd.read_csv(ni_path)
        vc = ni_df['label'].value_counts().sort_index()
        ax.bar(vc.index, vc.values,
               color=sns.color_palette('RdYlGn', len(vc)))
        ax.set_title('Distribution niveau_interet (train)', fontweight='bold')
        ax.set_ylabel('Nombre d\'exemples')
        ax.set_xlabel('Niveau (X/5)')
        for i, val in enumerate(vc.values):
            ax.text(i, val + 0.3, str(val), ha='center', fontweight='bold')

    plt.tight_layout()
    out_path = Path(out_base) / 'dataset_statistics.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Graphiques sauvegardés → {out_path} ✅")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARTIE 7 — RÉSUMÉ FINAL

════════════════════════════════════════════════════════════════════════════════

In [ ]:
def print_summary(ner_train, ner_val, ner_test,
                  qa_train, qa_val, qa_test,
                  clf_counts: dict) -> None:
    """Affiche un résumé récapitulatif de tous les datasets créés."""
    print("\n" + "=" * 60)
    print("RÉSUMÉ FINAL — DATASETS CRÉÉS")
    print("=" * 60)
    print(f"  NER   (CamemBERT + Flair) → {len(ner_train)} train | {len(ner_val)} val | {len(ner_test)} test")
    print(f"         Entités : MEDECIN, SPECIALITE, MEDICAMENT, GADGET")
    print(f"  QA    (QAmemberta)         → {len(qa_train)} train | {len(qa_val)} val | {len(qa_test)} test")
    print(f"         Champs : objectif, message_cle, reponse, action, commentaire")
    tv = clf_counts.get('train', {})
    print(f"  CLF   (CamemBERT Clf)      → type_visite: {tv.get('type_visite', 0)} | "
          f"niveau_interet: {tv.get('niveau_interet', 0)} train")
    print(f"         Champs : type_visite, niveau_interet")
    print("=" * 60)
    print("\n→ Lancer le Notebook 2 ensuite : entraînement des 3 modèles")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════

In [ ]:
# POINT D'ENTRÉE PRINCIPAL

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════

In [ ]:
if __name__ == '__main__':
    # 1. Chargement et nettoyage
    df, text_col = load_and_clean_csv(CSV_PATH, CONTEXT_COLS, GROUP_COL)

    # 2. Split par médecin (pas de fuite de données)
    train_df, val_df, test_df = split_by_group(df, GROUP_COL, TRAIN_SIZE, SEED)

    # 3. Construction du dataset NER
    ner_train, ner_val, ner_test = build_ner_dataset(
        train_df, val_df, test_df, OUT_BASE, text_col
    )

    # 4. Construction du dataset QA
    qa_train, qa_val, qa_test = build_qa_dataset(
        train_df, val_df, test_df, OUT_BASE, text_col
    )

    # 5. Construction du dataset Classification
    clf_counts = build_classification_dataset(
        train_df, val_df, test_df, OUT_BASE, text_col
    )

    # 6. Visualisations et statistiques
    plot_dataset_statistics(
        train_df, val_df, test_df, text_col,
        ner_train, ner_val, ner_test,
        qa_train, qa_val, qa_test,
        clf_counts,
        OUT_BASE,
    )

    # 7. Résumé final
    print_summary(ner_train, ner_val, ner_test, qa_train, qa_val, qa_test, clf_counts)